In [1]:
# =============================================================================
# PROJECT 2: SIMULATION-BASED OPTIMISATION — B-BICYCLE DIGITAL TWIN
# =============================================================================
#
# PREREQUISITES
#   Run Project 1 (digital_twin_trial.ipynb) with the P1 save block appended.
#   This produces the folder:   p1_artifacts/
#
# REPRODUCIBILITY — THREE SEED LEVELS
#   Level 1 — np.random.seed(SEED)
#       Controls the DES (BBicycleSimulator._run).  The DES uses the legacy
#       np.random module (Poisson arrivals, Normal service times, Bernoulli
#       rework).  Seeding once at startup makes every fresh run of this script
#       produce identical DES outputs for the same (C,R,Q,W,M1,M2) inputs
#       IN THE SAME CALL ORDER.  We do NOT re-seed inside any function so
#       that the seed is consumed deterministically from top to bottom.
#
#   Level 2 — RNG = np.random.default_rng(SEED)
#       A NEW-STYLE Generator, completely independent of the legacy
#       np.random stream.  Used exclusively for BO candidate sampling
#       (_random_candidates).  Kept module-level so the exact same candidate
#       pool is drawn each run regardless of how many DES calls occurred.
#
#   Level 3 — random_state=SEED  on every sklearn object
#       Ensures GP fitting, RF, GBR are deterministic.
#
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pickle
import json
import os
try:
    import joblib
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable,'-m','pip','install','joblib','--quiet'])
    import joblib
import warnings
from scipy import stats
from scipy.stats import norm as sp_norm
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

# GLOBAL CONSTANTS & SEEDS 

SEED           = 42
np.random.seed(SEED)                      # Level 1: DES stochasticity
RNG            = np.random.default_rng(SEED)  # Level 2: BO candidate sampling

P1_DIR         = "p1_artifacts"           # written by the P1 save block
OUT_DIR        = "p2_outputs"
os.makedirs(OUT_DIR, exist_ok=True)

MACHINE_BUDGET = 300_000                  # $300 K / month  →  M1+M2 ≤ 6
BO_PENALTY     = -5_000_000.0             # revenue assigned to infeasible candidates

# Decision-variable bounds  [lo, hi]  — order must match VAR_NAMES
VAR_BOUNDS = np.array([[0,1],[10,200],[10,200],[1,50],[1,5],[1,5]], dtype=float)
VAR_NAMES  = ['C','R','Q','W','M1','M2']


# SECTION 1  LOAD PROJECT 1 ARTIFACTS

from sklearn.preprocessing import StandardScaler

class EnsembleCalibrator:
    """Shell class matching P1 structure — enables joblib/pickle deserialization."""
    def __init__(self):
        self.rf                   = None
        self.gbr                  = None
        self.bayesian_ridge       = None
        self.scaler               = StandardScaler()
        self.sim_scaler           = StandardScaler()
        self.online_bias_history  = [0.0]
        self.online_scale_history = [1.0]
        self.feat_columns         = []


# calibrator is loaded with joblib (see load_p1_artifacts).
# joblib is the standard sklearn serialisation tool — it stores the class
# pickle tries to load a class saved from a Jupyter notebook into a script. Load everything written by the P1 save block.
# Nothing is re-trained; all models come from pickle.


def load_p1_artifacts():

    print("=" * 62)
    print("  LOADING PROJECT 1 ARTIFACTS")
    print("=" * 62)

    # Trained calibrator 
    # joblib avoids the __main__ class-name issue from pickle.
    cal = joblib.load(os.path.join(P1_DIR, "calibrator.joblib"))
    # Read converged Phase 2 params from JSON (history[-1]),
    # not from a potentially absent attribute on the original class.
    with open(os.path.join(P1_DIR, "p1_metrics.json")) as f:
        m = json.load(f)
    # Attaching as explicit scalars so evaluator code is clean
    cal.p2_bias  = m["p2_final_bias"]
    cal.p2_scale = m["p2_final_scale"]

    print(f"  calibrator.joblib   Phase2 bias=${cal.p2_bias:,.2f}  "
          f"scale={cal.p2_scale:.6f}")

    # ── Simulator theta 
    with open(os.path.join(P1_DIR, "simulator_theta.json")) as f:
        theta = json.load(f)
    print(f"  simulator_theta.json  ({len(theta)} params)")

    # ── Prediction arrays 
    arrs = {k: np.load(os.path.join(P1_DIR, f"{k}.npy"))
            for k in ["test_observed", "test_sim_baseline",
                      "test_p1_preds",   "test_p1_std",
                      "test_p2_preds",   "train_observed",
                      "train_sim_preds"]}
    n_test = len(arrs["test_observed"])
    print(f"  prediction arrays  (test={n_test} months)")

    # ── DataFrames 
    df_train = pd.read_csv(os.path.join(P1_DIR, "df_train.csv"))
    df_test  = pd.read_csv(os.path.join(P1_DIR, "df_test.csv"))
    print(f"  df_train({len(df_train)})  df_test({len(df_test)})")

    # ── Metrics JSON 
    print(f"  p1_metrics.json  "
          f"P1 R²={m['p1_test_r2']:.4f}  "
          f"RMSE=${m['p1_test_rmse']:,.0f}")
    print("=" * 62)

    return cal, theta, arrs, df_train, df_test, m

# SECTION 2  SIMULATOR  (identical to P1 — theta loaded from JSON).DES: Assembly → Cleaning → Inspection → [Pass/Fail → Rework loop]
# theta is loaded from p1_artifacts/simulator_theta.json.
# Uses the global np.random stream (seeded once at module level).

class BBicycleSimulator:

    def __init__(self, theta: dict):
        self.theta = theta

    def simulate(self, C, R, Q, W, M1, M2, n_days=30, n_reps=5):
        return float(np.mean(
            [self._run(int(C),int(R),int(Q),int(W),int(M1),int(M2),n_days)
             for _ in range(n_reps)]))

    def _run(self, C, R, Q, W, M1, M2, n_days):
        t = self.theta
        nr = -50000.0 * (M1 + M2)
        inv = 50
        for _ in range(n_days):
            for _ in range(np.random.poisson(t['arrival_rate'] * 24)):
                if inv <= R:
                    inv += Q;  nr -= 100.0 * Q
                if inv <= 0:
                    continue
                inv -= 1
                ft = (max(10, np.random.normal(t['assembly_mu'],   t['assembly_sigma']))
                    + max(5,  np.random.normal(t['cleaning_mu'],   t['cleaning_sigma']))   / max(M1, 1)
                    + max(3,  np.random.normal(t['inspection_mu'], t['inspection_sigma'])) / max(M2, 1))
                rn = 0
                while np.random.random() < t['fail_rate'] and rn < 3:
                    ft += max(10, np.random.normal(t['rework_mu'],     t['rework_sigma']))
                    ft += max(3,  np.random.normal(t['inspection_mu'], t['inspection_sigma'])) / max(M2, 1)
                    rn += 1;  nr -= 25.0
                fm  = ft / 60.0
                nr += (1000 - 5*fm) if C == 0 else (500 - 2*fm)
                nr -= 1.5 * (ft / 3600.0)
        return nr


# SECTION 3  FEATURE ENGINEERING  (exact copy of P1 — must not change)

def create_features(df):

    X    = df[['C','R','Q','W','M1','M2']].values.astype(float)
    feat = pd.DataFrame(X, columns=['C','R','Q','W','M1','M2'])
    feat['C_x_W']    = feat['C']  * feat['W']
    feat['R_x_Q']    = feat['R']  * feat['Q']
    feat['M1_x_M2']  = feat['M1'] * feat['M2']
    feat['M_total']  = feat['M1'] + feat['M2']
    feat['M_invest'] = 50000 * feat['M_total']
    feat['W_inv']    = 1.0 / (feat['W'] + 1)
    feat['R_over_Q'] = feat['R']  / (feat['Q'] + 1)
    feat['Q_x_W']    = feat['Q']  * feat['W']
    feat['C_x_M1']   = feat['C']  * feat['M1']
    feat['W_sq']     = feat['W']  ** 2
    feat['logW']     = np.log1p(feat['W'])
    feat['logR']     = np.log1p(feat['R'])
    feat['logQ']     = np.log1p(feat['Q'])
    return feat

# SECTION 4  DIGITAL TWIN EVALUATOR
#  p2_scale and p2_bias are read from the loaded calibrator's attached scalars
# (set by load_p1_artifacts from p1_metrics.json).
# They are NEVER updated during optimisation.

class DigitalTwinEvaluator:

    def __init__(self, cal, sim: BBicycleSimulator, sim_n_reps: int = 5):
        self.cal      = cal
        self.sim      = sim
        self.sim_reps = sim_n_reps
        self.n_calls  = 0
        self.call_log = []   # list of (C,R,Q,W,M1,M2, revenue, std)

    def evaluate(self, C, R, Q, W, M1, M2):
        # Step 1: DES — uses global np.random stream
        y_sim = self.sim.simulate(C, R, Q, W, M1, M2, n_reps=self.sim_reps)

        # Step 2: build feature row (no NetRevenue needed — dummy value)
        row = pd.DataFrame([{
            'C':C,'R':R,'Q':Q,'W':W,'M1':M1,'M2':M2,
            'NetRevenue':0.0,'Standard Deviation':20000.0
        }])
        feat              = create_features(row)
        feat['sim_pred']  = y_sim
        X                 = self.cal.scaler.transform(feat.values)
        rf_p              = self.cal.rf.predict(X)
        gbr_p             = self.cal.gbr.predict(X)
        stack             = np.column_stack([rf_p, gbr_p, np.array([y_sim])])
        mean_p1, std_br   = self.cal.bayesian_ridge.predict(stack, return_std=True)

        # RF tree uncertainty
        tree_p  = np.array([t.predict(X) for t in self.cal.rf.estimators_])
        rf_std  = float(np.std(tree_p, axis=0)[0])
        total_std = float(np.sqrt(float(std_br[0])**2 + rf_std**2))

        # Step 3: Phase 2 frozen correction
        revenue = float(self.cal.p2_scale * float(mean_p1[0]) + self.cal.p2_bias)

        self.n_calls += 1
        self.call_log.append((C,R,Q,W,M1,M2,revenue,total_std))
        return revenue, total_std

    def evaluate_scalar(self, C, R, Q, W, M1, M2): # Revenue only (used inside optimiser loop).
        rev, _ = self.evaluate(C, R, Q, W, M1, M2)
        return rev


# SECTION 5  BAYESIAN OPTIMISATION

def _normalise(X):
    return (X - VAR_BOUNDS[:,0]) / (VAR_BOUNDS[:,1] - VAR_BOUNDS[:,0])

def _random_candidates(n):

    return np.array([
        [RNG.integers(int(lo), int(hi)+1) for lo, hi in VAR_BOUNDS]
        for _ in range(n)
    ], dtype=float)

def _feasible(C, R, Q, W, M1, M2):
    return 50_000 * (int(M1) + int(M2)) <= MACHINE_BUDGET

def _expected_improvement(gp, X_cand, best_y, xi=0.01):
    mu, sig = gp.predict(_normalise(X_cand), return_std=True)
    sig     = np.maximum(sig, 1e-9)
    Z       = (mu - best_y - xi) / sig
    ei      = (mu - best_y - xi) * sp_norm.cdf(Z) + sig * sp_norm.pdf(Z)
    ei[sig < 1e-9] = 0.0
    return ei


def run_bayesian_optimisation(evaluator, n_calls=120,
                               n_initial=20, n_pool=2000, verbose=True):
    """
    GP-EI Bayesian Optimisation loop.

    Random phase  (iter 0 … n_initial-1):
        Candidates drawn uniformly by RNG; feasibility checked; first
        feasible candidate is evaluated.

    BO phase  (iter n_initial … n_calls-1):
        GP (Matérn-5/2 + WhiteKernel) fitted on all observations.
        EI computed over n_pool random candidates.
        Infeasible candidates get EI=0.
        Best-EI feasible candidate is evaluated.

    Constraint:  50,000*(M1+M2) ≤ 300,000  (i.e. M1+M2 ≤ 6)
    Infeasible evaluations receive BO_PENALTY so GP learns the boundary.

    Returns
    -------
    x_star       : dict   best feasible variable configuration
    best_history : list   best feasible revenue at each iteration
    df_log       : DataFrame  one row per evaluation
    """
    kernel = Matern(nu=2.5) + WhiteKernel(noise_level=1e-3,
                                           noise_level_bounds=(1e-6,1e-1))
    X_obs, y_obs = [], []
    best_y, best_x = BO_PENALTY, None
    best_history, log = [], []

    print("\n" + "=" * 62)
    print("  BAYESIAN OPTIMISATION — B-BICYCLE DIGITAL TWIN")
    print("=" * 62)
    print(f"  Budget      : {n_calls} evals "
          f"({n_initial} random + {n_calls-n_initial} BO-guided)")
    print(f"  Constraint  : M1+M2 ≤ 6  (budget $300K/month)")
    print(f"  Acquisition : Expected Improvement  ξ=0.01")
    print(f"  GP kernel   : Matérn-5/2 + WhiteKernel")
    print(f"  random_state: {SEED}  (GP fitting, sklearn)")
    print()

    for i in range(n_calls):
        # ── Candidate selection
        if i < n_initial:
            # Random phase — retry until feasible (max 500 tries)
            for _ in range(500):
                cand = _random_candidates(1)[0]
                if _feasible(*cand):
                    break
        else:
            # BO phase
            gp = GaussianProcessRegressor(
                kernel=kernel, normalize_y=True,
                n_restarts_optimizer=3, random_state=SEED   # Level 3 seed
            )
            Xa = np.array(X_obs);  ya = np.array(y_obs)
            ys = max(np.std(ya), 1.0)
            gp.fit(_normalise(Xa), ya / ys)

            pool = _random_candidates(n_pool)
            ei   = _expected_improvement(gp, pool, best_y / ys)
            # Zero out infeasible candidates
            for j, row in enumerate(pool):
                if not _feasible(*row):
                    ei[j] = 0.0
            cand = pool[np.argmax(ei)]

        # ── Evaluate
        C,R,Q,W,M1,M2 = (int(cand[0]),int(cand[1]),int(cand[2]),
                          int(cand[3]),int(cand[4]),int(cand[5]))
        feas = _feasible(C,R,Q,W,M1,M2)
        rev  = evaluator.evaluate_scalar(C,R,Q,W,M1,M2) if feas else BO_PENALTY

        X_obs.append(cand.copy());  y_obs.append(rev)
        if feas and rev > best_y:
            best_y = rev;  best_x = cand.copy()
        best_history.append(best_y)
        log.append(dict(iter=i+1, C=C, R=R, Q=Q, W=W, M1=M1, M2=M2,
                        revenue=rev if feas else np.nan,
                        feasible=feas, best_so_far=best_y))

        if verbose and (i+1) % 10 == 0:
            print(f"  Iter {i+1:>3d} | C={C} R={R:>3d} Q={Q:>3d} "
                  f"W={W:>2d} M1={M1} M2={M2} | "
                  f"Rev=${rev:>12,.0f} | Best=${best_y:>12,.0f}")

    x_star = {n: int(v) for n, v in zip(VAR_NAMES, best_x)}
    return x_star, best_history, pd.DataFrame(log)


# SECTION 6  SENSITIVITY ANALYSIS  (OAT)

def sensitivity_analysis(evaluator, x_star):
    """
    One-at-a-time sensitivity: sweep each variable across its full range
    while holding the others at x*.  Infeasible configurations get NaN.
    """
    print("\n  OAT sensitivity analysis ...")
    sweep = {
        'C':  ([0,1],                  "Contract Type"),
        'R':  (list(range(10,201,10)), "Reorder Point"),
        'Q':  (list(range(10,201,10)), "Order Quantity"),
        'W':  (list(range(1, 51, 2)),  "Max WIP"),
        'M1': ([1,2,3,4,5],           "Assembly Machines"),
        'M2': ([1,2,3,4,5],           "Rework Machines"),
    }
    base_rev = evaluator.evaluate_scalar(**x_star)
    results  = {}
    for var, (levels, label) in sweep.items():
        revs = []
        for val in levels:
            cfg = {**x_star, var: val}
            revs.append(evaluator.evaluate_scalar(**cfg)
                        if _feasible(**cfg) else np.nan)
        results[var] = {'label': label, 'levels': levels, 'revenues': revs}
        print(f"    {var:3s}  {label:<22s}  {len(levels)} levels")
    return base_rev, results

# SECTION 7  VALIDATION — REALITY GAP

# Run x* on the raw DES (physical system proxy) for n_reps months.
# Compare the resulting distribution against the twin prediction.
def validate_against_physical(sim: BBicycleSimulator, cal, x_star, n_reps=30):

    print("\n" + "=" * 62)
    print("  VALIDATION — REALITY GAP")
    print("=" * 62)
    C,R,Q,W,M1,M2 = (x_star['C'],x_star['R'],x_star['Q'],
                      x_star['W'],x_star['M1'],x_star['M2'])
    print(f"  x* = {x_star}")
    print(f"  Running {n_reps} DES replications ...")

    raw = [sim._run(C,R,Q,W,M1,M2,30) for _ in range(n_reps)]
    mu  = float(np.mean(raw));  sd = float(np.std(raw))
    ci  = stats.t.interval(0.95, df=n_reps-1, loc=mu, scale=stats.sem(raw))

    # Twin prediction at x* (use DES mean as sim_pred input)
    row = pd.DataFrame([{'C':C,'R':R,'Q':Q,'W':W,'M1':M1,'M2':M2,
                          'NetRevenue':0.0,'Standard Deviation':20000.0}])
    feat = create_features(row);  feat['sim_pred'] = mu
    X    = cal.scaler.transform(feat.values)
    rf_p = cal.rf.predict(X);  gbr_p = cal.gbr.predict(X)
    stk  = np.column_stack([rf_p, gbr_p, np.array([mu])])
    mp1, sbr = cal.bayesian_ridge.predict(stk, return_std=True)
    rf_std   = float(np.std(np.array([t.predict(X) for t in cal.rf.estimators_]),axis=0)[0])
    twin_std = float(np.sqrt(float(sbr[0])**2 + rf_std**2))
    twin     = float(cal.p2_scale * float(mp1[0]) + cal.p2_bias)

    gap_abs = twin - mu
    gap_pct = abs(gap_abs) / (abs(mu) + 1e-9) * 100
    inside  = ci[0] <= twin <= ci[1]

    print(f"\n  {'Metric':<44} {'Value':>16}")
    print(f"  {'-'*60}")
    print(f"  {'Physical mean  (DES, {n_reps} reps)':<44} ${mu:>15,.0f}")
    print(f"  {'Physical std':<44} ${sd:>15,.0f}")
    print(f"  {'Physical 95% CI':<44} "
          f"[${ci[0]:,.0f} — ${ci[1]:,.0f}]")
    print(f"  {'Digital Twin prediction':<44} ${twin:>15,.0f}")
    print(f"  {'Twin uncertainty  (1σ)':<44} ${twin_std:>15,.0f}")
    print(f"  {'Absolute gap  (Twin − Physical)':<44} ${gap_abs:>15,.0f}")
    print(f"  {'Relative gap  (%)':<44}  {gap_pct:>14.2f}%")
    print(f"  {'Twin inside Physical 95% CI':<44}  "
          f"{'YES ✓' if inside else 'NO  ✗':>14}")

    return dict(raw=raw, mu=mu, sd=sd, ci=ci,
                twin=twin, twin_std=twin_std,
                gap_abs=gap_abs, gap_pct=gap_pct, inside=inside)

# SECTION 8  VALUE OF SYNCHRONISATION

#  Coarse grid search using ONLY the raw uncalibrated DES.
#   Finds the DES-optimal config, then evaluates it on the calibrated twin.
#   Difference = economic value of Project 1 synchronization.

def value_of_synchronisation(evaluator, sim: BBicycleSimulator,
                               x_star, best_revenue):
    print("\n  Computing value of synchronisation ...")
    C_vals  = [0,1]
    M_pairs = [(1,1),(2,1),(1,2),(2,2),(3,1),(1,3),(3,2),(2,3),(3,3),
               (4,1),(1,4),(4,2),(2,4),(2,2),(5,1),(1,5)]
    R_vals  = [20,50,80,100,130,160,200]
    Q_vals  = [20,50,80,100,130,160,200]
    W_vals  = [5,10,20,30,40,50]

    best_raw_rev, best_raw_cfg = -np.inf, None
    for C in C_vals:
        for M1,M2 in M_pairs:
            if not _feasible(C,10,10,1,M1,M2): continue
            for R in R_vals:
                for Q in Q_vals:
                    for W in W_vals:
                        rev = sim.simulate(C,R,Q,W,M1,M2,n_reps=3)
                        if rev > best_raw_rev:
                            best_raw_rev = rev
                            best_raw_cfg = {'C':C,'R':R,'Q':Q,
                                            'W':W,'M1':M1,'M2':M2}

    twin_of_raw = evaluator.evaluate_scalar(**best_raw_cfg)
    gain        = best_revenue - twin_of_raw

    print(f"  Uncalibrated DES optimum : {best_raw_cfg}")
    print(f"  DES score of raw optimum : ${best_raw_rev:,.0f}")
    print(f"  Twin score of raw optimum: ${twin_of_raw:,.0f}")
    print(f"  Calibrated twin optimum  : ${best_revenue:,.0f}")
    print(f"  Gain from synchronisation: ${gain:,.0f}")

    return dict(raw_cfg=best_raw_cfg, raw_rev=best_raw_rev,
                twin_of_raw=twin_of_raw, gain=gain)


# SECTION 9  PLOTS

def _fmt_m(x, _):   return f'${x/1e6:.2f}M'
def _fmt_k(x, _):   return f'${x/1e3:.0f}K'

def plot_p1_recap(arrs, m):
    """Scatter plots re-drawn from loaded P1 arrays — no retraining."""
    ote = arrs['test_observed']
    fig, axes = plt.subplots(1,3, figsize=(16,5))
    sets = [("Baseline Simulation",   arrs['test_sim_baseline'],
             '#999999', m['baseline_r2'],  m['baseline_rmse']),
            ("Phase 1 Calibrated",    arrs['test_p1_preds'],
             '#2E5496', m['p1_test_r2'],   m['p1_test_rmse']),
            ("Phase 2 Online Bayesian",arrs['test_p2_preds'],
             '#1A5276', m['p2_test_r2'],   m['p2_test_rmse'])]
    for ax, (title, pred, col, r2, rmse) in zip(axes, sets):
        vmin = min(ote.min(), pred.min()); vmax = max(ote.max(), pred.max())
        ax.scatter(ote, pred, alpha=0.5, s=18, color=col)
        ax.plot([vmin,vmax],[vmin,vmax],'r--',lw=1.2)
        ax.set_title(f'{title}\nRMSE=${rmse:,.0f}  R²={r2:.3f}',
                     fontsize=10, fontweight='bold')
        ax.set_xlabel('Observed ($)'); ax.set_ylabel('Predicted ($)')
        ax.grid(True, alpha=0.3)
        ax.xaxis.set_major_formatter(plt.FuncFormatter(_fmt_m))
        ax.yaxis.set_major_formatter(plt.FuncFormatter(_fmt_m))
    plt.suptitle('Project 1 Digital Twin — Test-Set Performance (loaded from artifacts)',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    _save('fig_p2_00_p1_recap.png')


def plot_convergence(best_history, df_log, x_star, best_revenue, n_initial):
    fig, axes = plt.subplots(1,2, figsize=(14,5))

    ax = axes[0]
    ax.plot(best_history, color='#2E5496', lw=2)
    ax.axvline(n_initial, color='gray', ls=':', lw=1.2, label='BO phase starts')
    ax.axhline(best_revenue, color='#C00000', ls='--', lw=1.5,
               label=f'Optimum = ${best_revenue:,.0f}')
    ax.fill_between(range(len(best_history)), best_history, alpha=0.12, color='#2E5496')
    ax.set_xlabel('BO Iteration', fontsize=11)
    ax.set_ylabel('Best Revenue Found', fontsize=11)
    ax.set_title('Convergence: Best Revenue vs. Iteration', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.yaxis.set_major_formatter(plt.FuncFormatter(_fmt_m))

    ax2 = axes[1]
    feas   = df_log[df_log['feasible']]
    infeas = df_log[~df_log['feasible']]
    ax2.scatter(feas['iter'],   feas['revenue'],    alpha=0.5, s=18,
                color='#2E5496', label='Feasible')
    if len(infeas):
        ax2.scatter(infeas['iter'], [0]*len(infeas), alpha=0.3, s=12,
                    color='gray', marker='x', label='Infeasible')
    ax2.axhline(best_revenue, color='#C00000', ls='--', lw=1.5,
                label=f'Optimum = ${best_revenue:,.0f}')
    ax2.set_xlabel('BO Iteration', fontsize=11)
    ax2.set_ylabel('Evaluated Revenue', fontsize=11)
    ax2.set_title('All Candidate Evaluations', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=9); ax2.grid(True, alpha=0.3)
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(_fmt_m))

    plt.tight_layout()
    _save('fig_p2_01_convergence.png')


def plot_sensitivity(base_rev, sens, x_star):
    fig, axes = plt.subplots(2,3, figsize=(16,9))
    for ax, var in zip(axes.flatten(), sens):
        info   = sens[var]
        levels = info['levels']
        revs   = [r if not np.isnan(r) else 0 for r in info['revenues']]
        cols   = ['#AAAAAA' if np.isnan(info['revenues'][j]) else '#2E5496'
                  for j in range(len(levels))]
        ax.bar(range(len(levels)), revs, color=cols, alpha=0.8)
        ax.axhline(base_rev, color='#C00000', ls='--', lw=1.5, label='x* optimum')
        if x_star[var] in levels:
            oi = levels.index(x_star[var])
            ax.bar(oi, revs[oi], color='#C00000', alpha=1.0, label=f'x*={x_star[var]}')
        ax.set_xticks(range(len(levels)))
        ax.set_xticklabels([str(l) for l in levels], fontsize=8, rotation=45)
        ax.set_title(f"{info['label']} ({var})", fontweight='bold')
        ax.set_ylabel('Revenue ($)')
        ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')
        ax.yaxis.set_major_formatter(plt.FuncFormatter(_fmt_m))
    plt.suptitle('Sensitivity Analysis (OAT): Revenue vs. Each Decision Variable',
                 fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout()
    _save('fig_p2_02_sensitivity.png')


def plot_validation(val, x_star):
    fig, axes = plt.subplots(1,2, figsize=(13,5))

    ax = axes[0]
    ax.hist(val['raw'], bins=12, color='#2E5496', alpha=0.75, edgecolor='white')
    ax.axvline(val['mu'],   color='navy',    lw=2, label=f"Physical mean = ${val['mu']:,.0f}")
    ax.axvline(val['twin'], color='#C00000', lw=2, ls='--',
               label=f"Twin pred = ${val['twin']:,.0f}")
    ax.axvline(val['ci'][0], color='gray', lw=1, ls=':')
    ax.axvline(val['ci'][1], color='gray', lw=1, ls=':', label='Physical 95% CI')
    ax.set_xlabel('Net Revenue ($)'); ax.set_ylabel('Count')
    ax.set_title('Physical System Distribution at x*', fontsize=12, fontweight='bold')
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(_fmt_m))

    ax2 = axes[1]
    vals_  = [val['mu'], val['twin']]
    bars   = ax2.bar(['Physical\n(DES mean)','Digital Twin\n(Phase 1+2)'],
                     vals_, color=['#2E5496','#C00000'], alpha=0.85, width=0.4)
    ci_h   = (val['ci'][1]-val['ci'][0])/2
    ax2.errorbar([0],[val['mu']],   yerr=ci_h,           fmt='none',
                 color='black', capsize=8, lw=2)
    ax2.errorbar([1],[val['twin']], yerr=1.96*val['twin_std'], fmt='none',
                 color='black', capsize=8, lw=2)
    for bar, v in zip(bars, vals_):
        ax2.text(bar.get_x()+bar.get_width()/2, v+max(vals_)*0.005,
                 f'${v:,.0f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax2.set_ylabel('Net Revenue ($)')
    ax2.set_title(f'Reality Gap: {val["gap_pct"]:.2f}%  (${val["gap_abs"]:,.0f})',
                  fontsize=12, fontweight='bold')
    ax2.grid(True, alpha=0.3, axis='y')
    ax2.yaxis.set_major_formatter(plt.FuncFormatter(_fmt_m))

    plt.tight_layout()
    _save('fig_p2_03_validation.png')


def plot_value_of_sync(sync, best_revenue):
    fig, ax = plt.subplots(figsize=(10,5))
    labels = ['Uncalibrated DES\nOptimum\n(DES score)',
              'Uncalibrated DES\nOptimum\n(Twin score)',
              'Calibrated Twin\nOptimum\n(Twin score)']
    vals   = [sync['raw_rev'], sync['twin_of_raw'], best_revenue]
    bars   = ax.bar(labels, vals, color=['#AAAAAA','#7090C0','#2E5496'],
                    alpha=0.88, width=0.5)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2,
                v+(max(vals)-min(vals))*0.01,
                f'${v:,.0f}', ha='center', va='bottom',
                fontsize=10, fontweight='bold')
    # Arrow showing gain
    ax.annotate('', xy=(2, best_revenue), xytext=(1, sync['twin_of_raw']),
                arrowprops=dict(arrowstyle='<->', color='#C00000', lw=2))
    ax.text(1.5, (best_revenue+sync['twin_of_raw'])/2,
            f" Gain = ${sync['gain']:,.0f}",
            ha='left', va='center', color='#C00000', fontweight='bold')
    ax.set_ylabel('Net Revenue ($)')
    ax.set_title('Value of Digital Twin Synchronisation\n'
                 'Calibrated vs Uncalibrated Optimisation',
                 fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    ax.yaxis.set_major_formatter(plt.FuncFormatter(_fmt_m))
    plt.tight_layout()
    _save('fig_p2_04_value_of_sync.png')


def _save(fname):
    path = os.path.join(OUT_DIR, fname)
    plt.savefig(path, dpi=200, bbox_inches='tight')
    plt.close()
    print(f"  Saved: {path}")


# SECTION 10  SAVE P2 RESULTS

def save_p2_results(x_star, best_revenue, best_std,
                    best_history, val, sync, df_log, m):
    summary = {
        # ── Optimal solution
        'x_star':                 x_star,
        'best_revenue_twin':      float(best_revenue),
        'best_revenue_std':       float(best_std),
        'best_revenue_ci95_lo':   float(best_revenue - 1.96*best_std),
        'best_revenue_ci95_hi':   float(best_revenue + 1.96*best_std),
        'machine_cost_per_month': 50_000*(x_star['M1']+x_star['M2']),

        # ── Comparison to historical baseline
        'hist_mean':              m['field_revenue_mean'],
        'hist_max':               m['field_revenue_max'],
        'improvement_abs':        float(best_revenue - m['field_revenue_mean']),
        'improvement_pct':        float((best_revenue/m['field_revenue_mean']-1)*100),

        # ── Validation
        'physical_mean':          val['mu'],
        'physical_std':           val['sd'],
        'physical_ci95_lo':       float(val['ci'][0]),
        'physical_ci95_hi':       float(val['ci'][1]),
        'gap_abs':                val['gap_abs'],
        'gap_pct':                val['gap_pct'],
        'twin_inside_physical_ci':bool(val['inside']),

        # ── Value of synchronisation
        'uncalib_cfg':            sync['raw_cfg'],
        'uncalib_raw_rev':        float(sync['raw_rev']),
        'uncalib_twin_rev':       float(sync['twin_of_raw']),
        'sync_gain':              float(sync['gain']),

        # ── BO run info
        'bo_total_evals':         int(df_log.shape[0]),
        'bo_feasible_evals':      int(df_log['feasible'].sum()),
        'convergence_last10':     [float(v) for v in best_history[-10:]],

        # ── P1 provenance (carried forward for report)
        'p1_test_r2':             m['p1_test_r2'],
        'p1_test_rmse':           m['p1_test_rmse'],
        'p2_final_bias':          m['p2_final_bias'],
        'p2_final_scale':         m['p2_final_scale'],
    }
    path = os.path.join(OUT_DIR, 'p2_results.json')
    with open(path, 'w') as f:
        json.dump(summary, f, indent=2)
    print(f"  Saved: {path}")
    return summary

# MAIN

def main():
    print("\n" + "=" * 62)
    print("  PROJECT 2 — SIMULATION-BASED OPTIMISATION")
    print("  B-Bicycle Digital Twin")
    print("=" * 62)

    # 1. Load all P1 artifacts — NO retraining
    cal, theta, arrs, df_train, df_test, m = load_p1_artifacts()
    sim       = BBicycleSimulator(theta)
    evaluator = DigitalTwinEvaluator(cal, sim, sim_n_reps=3)

    # 2. P1 recap figure (from loaded arrays, not recomputed)
    print("\n  Generating P1 recap figure ...")
    plot_p1_recap(arrs, m)

    # 3. Bayesian Optimisation
    N_INITIAL = 20
    x_star, best_history, df_log = run_bayesian_optimisation(
        evaluator, n_calls=120, n_initial=N_INITIAL, verbose=True
    )
    best_revenue, best_std = evaluator.evaluate(**x_star)

    print("\n" + "=" * 62)
    print("  OPTIMAL SOLUTION x*")
    print("=" * 62)
    print(f"  C  = {x_star['C']}  ({'Standard' if x_star['C']==0 else 'Expedited'})")
    print(f"  R  = {x_star['R']}")
    print(f"  Q  = {x_star['Q']}")
    print(f"  W  = {x_star['W']}")
    print(f"  M1 = {x_star['M1']}")
    print(f"  M2 = {x_star['M2']}")
    print(f"  Machine investment = ${50_000*(x_star['M1']+x_star['M2']):,}/month")
    print(f"  Twin revenue       = ${best_revenue:,.0f}  "
          f"(95% CI ${best_revenue-1.96*best_std:,.0f} — "
          f"${best_revenue+1.96*best_std:,.0f})")
    print(f"  Historical mean    = ${m['field_revenue_mean']:,.0f}")
    print(f"  Improvement        = ${best_revenue-m['field_revenue_mean']:,.0f}  "
          f"(+{(best_revenue/m['field_revenue_mean']-1)*100:.1f}%)")

    # 4. Sensitivity analysis
    base_rev, sens = sensitivity_analysis(evaluator, x_star)

    # 5. Validation — reality gap
    val = validate_against_physical(sim, cal, x_star, n_reps=30)

    # 6. Value of synchronisation
    sync = value_of_synchronisation(evaluator, sim, x_star, best_revenue)

    # 7. All plots
    print("\n  Generating figures ...")
    plot_convergence(best_history, df_log, x_star, best_revenue, N_INITIAL)
    plot_sensitivity(base_rev, sens, x_star)
    plot_validation(val, x_star)
    plot_value_of_sync(sync, best_revenue)

    # 8. Save JSON results
    summary = save_p2_results(
        x_star, best_revenue, best_std,
        best_history, val, sync, df_log, m
    )

    # 9. Final summary
    print("\n" + "=" * 62)
    print("  FINAL SUMMARY")
    print("=" * 62)
    print(f"  DT evaluations      : {evaluator.n_calls}")
    print(f"  x*                  : {x_star}")
    print(f"  Twin revenue        : ${best_revenue:,.0f}")
    print(f"  Physical revenue    : ${val['mu']:,.0f}")
    print(f"  Reality gap         : {val['gap_pct']:.2f}%")
    print(f"  Sync gain           : ${sync['gain']:,.0f}")
    print(f"\n  Output → {os.path.abspath(OUT_DIR)}/")
    for f in sorted(os.listdir(OUT_DIR)):
        print(f"    {f}")


if __name__ == '__main__':
    main()





  PROJECT 2 — SIMULATION-BASED OPTIMISATION
  B-Bicycle Digital Twin
  LOADING PROJECT 1 ARTIFACTS
  calibrator.joblib   Phase2 bias=$26,269.44  scale=1.036190
  simulator_theta.json  (10 params)
  prediction arrays  (test=72 months)
  df_train(288)  df_test(72)
  p1_metrics.json  P1 R²=0.9583  RMSE=$163,832

  Generating P1 recap figure ...
  Saved: p2_outputs/fig_p2_00_p1_recap.png

  BAYESIAN OPTIMISATION — B-BICYCLE DIGITAL TWIN
  Budget      : 120 evals (20 random + 100 BO-guided)
  Constraint  : M1+M2 ≤ 6  (budget $300K/month)
  Acquisition : Expected Improvement  ξ=0.01
  GP kernel   : Matérn-5/2 + WhiteKernel
  random_state: 42  (GP fitting, sklearn)

  Iter  10 | C=1 R=137 Q=189 W=22 M1=1 M2=5 | Rev=$  -1,027,679 | Best=$     720,603
  Iter  20 | C=1 R= 79 Q=193 W= 5 M1=2 M2=1 | Rev=$    -253,028 | Best=$   1,492,917
  Iter  30 | C=0 R= 18 Q= 86 W=45 M1=1 M2=5 | Rev=$    -605,430 | Best=$   2,695,554
  Iter  40 | C=0 R= 90 Q= 10 W= 1 M1=4 M2=2 | Rev=$   2,355,962 | Best=$   2